# SFT Fine-Tuning on Ray with Training Hub on OpenShift AI

This notebook runs **Supervised Fine-Tuning (SFT)** using Training Hub on a Ray cluster.
We train [Qwen/Qwen2.5-1.5B-Instruct](https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct)
to produce structured JSON output using the
[Table-GPT](https://huggingface.co/datasets/LipengCS/Table-GPT) dataset.

The Training Hub `sft()` function is called directly in a RayJob entrypoint.
CodeFlare SDK submits the job as a `RayJob` with a `ManagedClusterConfig` — the SDK
creates a short-lived RayCluster, runs the job, and tears everything down on completion.

For more details on SFT and hardware requirements, see the [README](./README.md).

## Setup

Import the required dependencies.

In [ ]:
from codeflare_sdk import ManagedClusterConfig, RayJob
from kubernetes.client import (
    V1PersistentVolumeClaimVolumeSource,
    V1Volume,
    V1VolumeMount,
)

## Authenticate to your OpenShift Cluster

Log in via `oc login`. Get your token from `oc whoami -t` or the OpenShift console
(Copy login command).

> **Note:** The CodeFlare SDK reads authentication from the local kubeconfig file.
> Using `oc login` ensures the kubeconfig is written correctly for the SDK to pick up.

In [ ]:
!oc login --token=<YOUR_TOKEN> --server=<YOUR_API_SERVER> --insecure-skip-tls-verify

## 1. Configure Training Parameters

Update the following variables to match your environment.

### Cluster Settings
- **NAMESPACE**: Your OpenShift project/namespace
- **IMAGE**: The Ray runtime image with Training Hub pre-installed

### Storage
- **PVC_NAME**: Name of the PVC containing the model and dataset
- **PVC_MOUNT_PATH**: Mount path inside the Ray pods

### Model and Data
- **MODEL_PATH**: Path to the model on the PVC
- **DATA_PATH**: Path to the training dataset (JSONL format) on the PVC
- **CKPT_DIR**: Path for training checkpoints

### SFT Hyperparameters
- **MAX_SEQ_LEN**: Maximum sequence length for tokenization
- **MAX_TOKENS_PER_GPU**: Maximum token batch size per GPU
- **EFFECTIVE_BATCH_SIZE**: Effective batch size (gradient accumulation adjusts automatically)
- **LEARNING_RATE**: Learning rate for the optimizer
- **NUM_EPOCHS**: Number of training epochs

In [ ]:
# Cluster settings
NAMESPACE = "<YOUR_NAMESPACE>"
IMAGE = "quay.io/modh/ray@sha256:f63a015302758f805e5332605669b886e4d7ac60ec929413a2ffc19a904211c6"  # quay.io/modh/ray:2.55.1-py312-cu129-th081

# HuggingFace token (optional — only needed for gated models)
HF_TOKEN = ""

# Storage — PVC with model and training data
PVC_NAME = "shared"  # Replace if the shared RWX storage name is different than in the example provided
PVC_PATH = "shared"  # Replace if the shared RWX storage path is different than in the example provided
PVC_MOUNT_PATH = f"/opt/app-root/src/{PVC_PATH}"

# Model and data paths
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
MODEL_PATH = f"{PVC_MOUNT_PATH}/{MODEL_ID}"
DATA_PATH = f"{PVC_MOUNT_PATH}/table-gpt-data/train/train_All_5000.jsonl"
CKPT_DIR = f"{PVC_MOUNT_PATH}/checkpoints/sft"

# SFT hyperparameters
MAX_SEQ_LEN = 512
MAX_TOKENS_PER_GPU = 512
EFFECTIVE_BATCH_SIZE = 4
LEARNING_RATE = 1e-5
NUM_EPOCHS = 1

print(f"Namespace:  {NAMESPACE}")
print(f"Image:      {IMAGE}")
print(f"Model:      {MODEL_PATH}")
print(f"Data:       {DATA_PATH}")

## 2. Download Model and Prepare Dataset

Download the model to the shared PVC and prepare the training dataset.

We use the [Table-GPT](https://huggingface.co/datasets/LipengCS/Table-GPT) dataset
from Microsoft, which trains the model to answer questions about tabular data in
structured JSON format.

In [ ]:
import os

from huggingface_hub import snapshot_download

model_dir = f"/opt/app-root/src/{PVC_PATH}/{MODEL_ID}"

if os.path.exists(model_dir) and os.listdir(model_dir):
    print(f"Model already exists at {model_dir}, skipping download")
else:
    print(f"Downloading {MODEL_ID} to {model_dir}...")
    snapshot_download(
        repo_id=MODEL_ID,
        local_dir=model_dir,
        token=HF_TOKEN or True,
        resume_download=True,
        local_dir_use_symlinks=False,
    )
    print(f"Model downloaded to {model_dir}")

In [ ]:
import json
import random

from datasets import load_dataset

print("Loading Table-GPT dataset...")
dataset = load_dataset("LipengCS/Table-GPT", "All")

train_data = dataset["train"]
print(f"Original training set size: {len(train_data)}")

random.seed(42)
subset_indices = random.sample(range(len(train_data)), min(5000, len(train_data)))
subset_data = train_data.select(subset_indices)
print(f"Subset size: {len(subset_data)}")

output_dir = f"/opt/app-root/src/{PVC_PATH}/table-gpt-data/train"
os.makedirs(output_dir, exist_ok=True)

output_file = f"{output_dir}/train_All_5000.jsonl"
with open(output_file, "w") as f:
    for example in subset_data:
        f.write(json.dumps(example) + "\n")

print(f"Dataset saved to {output_file}")

## 3. Configure the Ray Cluster

SFT runs as a single GPU job — all training happens on the head pod.

The PVC is mounted on the head pod so the model and dataset are accessible.
No worker pods are needed for single GPU SFT.

In [ ]:
pvc_volume = V1Volume(
    name="training-data",
    persistent_volume_claim=V1PersistentVolumeClaimVolumeSource(claim_name=PVC_NAME),
)
pvc_mount = V1VolumeMount(name="training-data", mount_path=PVC_MOUNT_PATH)

env_vars = {}
if HF_TOKEN:
    env_vars["HF_TOKEN"] = HF_TOKEN
    env_vars["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN

cluster_config = ManagedClusterConfig(
    image=IMAGE,
    num_workers=0,
    head_cpu_requests=4,
    head_cpu_limits=8,
    head_memory_requests=64,
    head_memory_limits=64,
    head_accelerators={"nvidia.com/gpu": 1},
    volumes=[pvc_volume],
    volume_mounts=[pvc_mount],
    envs=env_vars,
)

print("Cluster config: head pod only (single GPU SFT)")
print("  GPU: 1× NVIDIA")
print(f"  PVC: {PVC_NAME} mounted at {PVC_MOUNT_PATH}")

## 4. Build the Training Entrypoint

The entrypoint calls `training_hub.sft()` directly.

In [ ]:
entrypoint = f'''python -c "
from training_hub import sft

sft(
    model_path='{MODEL_PATH}',
    data_path='{DATA_PATH}',
    ckpt_output_dir='{CKPT_DIR}',
    data_output_dir='{CKPT_DIR}/data',
    max_seq_len={MAX_SEQ_LEN},
    max_tokens_per_gpu={MAX_TOKENS_PER_GPU},
    effective_batch_size={EFFECTIVE_BATCH_SIZE},
    learning_rate={LEARNING_RATE},
    num_epochs={NUM_EPOCHS},
)

print('SFT training completed successfully')
"'''

print("Entrypoint configured")
print(f"  Model: {MODEL_PATH}")
print(f"  Data:  {DATA_PATH}")
print(f"  Epochs: {NUM_EPOCHS}")

## 5. Submit the RayJob

The SDK creates a `RayJob` CR with an embedded `RayCluster` spec. KubeRay:
1. Spins up the Ray cluster (head pod with GPU)
2. Runs the entrypoint on the head node
3. Tears everything down when the job finishes

The `ttl_seconds_after_finished` controls how long the RayJob CR persists after
completion (for log inspection). Set to 0 for immediate cleanup.

In [ ]:
job = RayJob(
    job_name="sft-training-hub",
    entrypoint=entrypoint,
    cluster_config=cluster_config,
    namespace=NAMESPACE,
    ttl_seconds_after_finished=600,
)

job.submit()
print(f"RayJob '{job.name}' submitted to namespace '{NAMESPACE}'")

## 6. Monitor Training Progress

Poll the job status. The status may show as `PENDING` while the RayCluster is
spinning up and pulling the image.

SFT training with the default parameters on a single L40S (48GB) takes approximately
10-20 minutes depending on dataset size.

In [ ]:
job.status()

## 7. Cleanup

The RayCluster is automatically torn down when the job finishes.
The RayJob CR is cleaned up after `ttl_seconds_after_finished` (600 seconds).
Training checkpoints are persisted on the PVC and remain available after cleanup.

Use `job.delete()` for immediate cleanup if needed.

In [ ]:
# Uncomment to delete the RayJob immediately:
# job.delete()

## 8. Test the Trained Model

With checkpoints saved to the PVC, we can load the trained model directly in this
workbench and run inference. This verifies the model learned the expected behavior.

In [ ]:
import glob
import os


def find_most_recent_checkpoint(output_dir):
    """
    Find the most recent checkpoint in the training output directory.

    Checks for HuggingFace Trainer format (checkpoint-{step}/) and
    hf_format/samples_* layout.

    Args:
        output_dir (str): Training output directory

    Returns:
        str: Path to the most recent checkpoint

    Raises:
        ValueError: If no checkpoints are found
    """
    patterns = [
        os.path.join(output_dir, "hf_format", "samples_*"),
        os.path.join(output_dir, "checkpoint-*"),
    ]

    for pattern in patterns:
        checkpoint_dirs = glob.glob(pattern)
        if checkpoint_dirs:
            return max(checkpoint_dirs, key=os.path.getctime)

    raise ValueError(
        f"No checkpoints found in {output_dir}. "
        f"Contents: {os.listdir(output_dir) if os.path.exists(output_dir) else 'directory not found'}"
    )


print("Checkpoint utility functions defined")

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

final_checkpoint = find_most_recent_checkpoint(CKPT_DIR)
print(f"Loading checkpoint: {final_checkpoint}")

use_cuda = torch.cuda.is_available()
load_kwargs = {
    "dtype": torch.bfloat16 if use_cuda else torch.float32,
    "device_map": "cuda:0" if use_cuda else "cpu",
    "attn_implementation": "sdpa" if use_cuda else "eager",
    "trust_remote_code": True,
}

print(f"Loading the trained model... ({'CUDA' if use_cuda else 'CPU'} mode)")
trained_tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
trained_model = AutoModelForCausalLM.from_pretrained(
    final_checkpoint,
    **load_kwargs,
)

print("Model loaded successfully")
print(f"Model parameters: {trained_model.num_parameters():,}")

In [ ]:
user_message = """# Task Description: Please look at the table, and then answer the question. Return the final result as JSON in the format {"answer": "<YOUR ANSWER>"}.

## Input:
*Table*
|Rank|Cyclist|Team|Time|UCI ProTour\nPoints|
|---|---|---|---|---|
|1|Alejandro Valverde (ESP)|Caisse d'Epargne|5h 29' 10"|40|
|2|Alexandr Kolobnev (RUS)|Team CSC Saxo Bank|s.t.|30|
|3|Davide Rebellin (ITA)|Gerolsteiner|s.t.|25|
|4|Paolo Bettini (ITA)|Quick Step|s.t.|20|
|5|Franco Pellizotti (ITA)|Liquigas|s.t.|15|
|6|Denis Menchov (RUS)|Rabobank|s.t.|11|
|7|Samuel Sánchez (ESP)|Euskaltel-Euskadi|s.t.|7|
|8|Stéphane Goubert (FRA)|Ag2r-La Mondiale|+ 2"|5|
|9|Haimar Zubeldia (ESP)|Euskaltel-Euskadi|+ 2"|3|
|10|David Moncoutié (FRA)|Cofidis|+ 2"|1|
*Question:*
which country had the most cyclists finish within the top 10?

Return the final result as JSON in the format {"answer": "<YOUR ANSWER>"}.
## Output:"""

if hasattr(trained_tokenizer, "apply_chat_template"):
    messages = [{"role": "user", "content": user_message}]
    formatted_input = trained_tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
else:
    formatted_input = f"User: {user_message}\nAssistant:"

print("Testing model with table reasoning task:")
print("Question: Which country had the most cyclists finish within the top 10?")

inputs = trained_tokenizer(formatted_input, return_tensors="pt").to(
    trained_model.device
)

with torch.no_grad():
    outputs = trained_model.generate(
        **inputs,
        max_new_tokens=256,
        do_sample=True,
        temperature=0.1,
        top_p=0.95,
    )

response = trained_tokenizer.decode(outputs[0], skip_special_tokens=True)
if formatted_input in response:
    response = response.replace(formatted_input, "").strip()

print(f"\nModel response:\n{response}")
print("\nTable reasoning test completed!")

## Summary

This notebook demonstrated end-to-end SFT fine-tuning on Ray with Training Hub:

1. **Configured** a single GPU RayJob with PVC storage
2. **Trained** Qwen 2.5 1.5B Instruct on structured output using Training Hub's `sft()` function
3. **Evaluated** the trained model on a table reasoning task

The trained model checkpoints are persisted on the PVC at the configured `CKPT_DIR` path
and can be used for serving or further fine-tuning.